# Lecture 12: Transformer Models & Libraries — Answer Key
### NLP Course 2027 — Week 06

---
Complete answers for all four exercises in Lecture 12.

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, pipeline
from sklearn.metrics.pairwise import cosine_similarity
print('Ready')

---
## Exercise 12.1 — Summarization: BART vs PEGASUS

**Task:** Compare `facebook/bart-large-cnn` and `google/pegasus-xsum` on a long article.

**Key concept:** BART (denoising autoencoder) is trained on noisy-document reconstruction; it produces longer, more faithful summaries. PEGASUS (gap-sentence generation) is trained for extreme summarization; it produces shorter, more abstractive summaries.

In [ ]:
long_article = """
Artificial intelligence is transforming every industry on the planet. From healthcare
to finance, education to transportation, AI systems are being deployed at unprecedented
scale. In healthcare, machine learning models now diagnose diseases from medical images
with accuracy matching specialist physicians. Radiologists use AI to detect early-stage
cancers, cardiologists rely on algorithms to analyze ECG patterns, and pathologists
employ deep learning to identify cellular abnormalities. In finance, AI drives automated
trading, fraud detection, credit scoring, and customer service chatbots. Banks save
billions of dollars annually through AI-powered risk assessment. The transportation
sector sees AI enabling semi-autonomous and fully autonomous vehicles. Companies like
Waymo, Tesla, and Cruise have logged millions of autonomous miles. In education, adaptive
learning platforms personalize content for each student. However, AI also brings
significant challenges. Algorithmic bias can perpetuate societal inequalities. Privacy
concerns arise as AI systems require vast amounts of personal data. The EU AI Act
represents the world's first comprehensive AI regulation framework.
"""

try:
    bart_pipe = pipeline('summarization', model='facebook/bart-large-cnn')
    bart_out  = bart_pipe(long_article, max_length=120, min_length=40, do_sample=False)
    print('BART (bart-large-cnn):')
    print(bart_out[0]['summary_text'])
    print(f'Length: {len(bart_out[0]["summary_text"].split())} words\n')
except Exception as e:
    print(f'BART unavailable: {e}')

try:
    peg_pipe = pipeline('summarization', model='google/pegasus-xsum')
    peg_out  = peg_pipe(long_article, max_length=60, do_sample=False)
    print('PEGASUS (pegasus-xsum):')
    print(peg_out[0]['summary_text'])
    print(f'Length: {len(peg_out[0]["summary_text"].split())} words')
except Exception as e:
    print(f'PEGASUS unavailable: {e}')

print()
print('Expected difference:')
print('  BART: longer, more extractive, faithful to source sentences')
print('  PEGASUS: shorter, more abstractive, may rephrase significantly')

---
## Exercise 12.2 — Tokenizer Comparison

**Task:** Tokenize words with BERT, GPT-2, and T5 tokenizers. Compare subword splits.

**Key concept:** All three use subword tokenization but different algorithms: BERT uses WordPiece (## prefix for continuations), GPT-2 uses BPE (Ġ prefix for space), T5 uses SentencePiece (▁ prefix for word-start). Unknown/rare words get split more aggressively.

In [ ]:
words = ['unhappiness', 'tokenization', 'COVID-19', 'transformer', 'unbelievable']

tokenizers = {
    'BERT':  'bert-base-uncased',
    'GPT-2': 'gpt2',
    'T5':    't5-small',
}

loaded = {name: AutoTokenizer.from_pretrained(ckpt)
          for name, ckpt in tokenizers.items()}

print(f'{"Word":<18}', '  '.join(f'{n:<28}' for n in tokenizers))
print('-' * 100)
for word in words:
    row = f'{word:<18}'
    for name, tok in loaded.items():
        tokens = tok.tokenize(word)
        row += f'  {str(tokens):<28}'
    print(row)

print()
print('Prefix conventions:')
print('  BERT:  ## marks continuation subwords (unhappiness -> un ##happiness)')
print('  GPT-2: Ġ marks word-start (space before word)')
print('  T5:    ▁ marks word-start (SentencePiece unigram model)')

---
## Exercise 12.3 — Zero-Shot Classification

**Task:** Classify 10 customer reviews into 4 categories. Compute accuracy.

**Key concept:** Zero-shot classification uses NLI (Natural Language Inference): it treats each label as a hypothesis and scores whether the text entails it. No task-specific training needed.

In [ ]:
reviews_with_labels = [
    ('The item arrived broken and poorly packaged.',       'product quality'),
    ('Delivery took 3 weeks, way too slow.',               'shipping'),
    ('The support team resolved my issue in 5 minutes!',  'customer service'),
    ('Too expensive compared to similar products.',        'pricing'),
    ('Amazing build quality, exactly as described.',      'product quality'),
    ('Express shipping was worth the extra cost.',         'shipping'),
    ('Rude customer service rep refused to help.',         'customer service'),
    ('Great value for money.',                             'pricing'),
    ('Product stopped working after one week.',            'product quality'),
    ('Free shipping threshold is too high.',               'shipping'),
]
candidate_labels = ['product quality', 'shipping', 'customer service', 'pricing']

try:
    zsc = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
    correct = 0
    print(f'{"Review":<50} {"True":<18} {"Predicted":<18} {"OK"}')
    print('-' * 95)
    for review, true_label in reviews_with_labels:
        result    = zsc(review, candidate_labels)
        predicted = result['labels'][0]
        ok = '✓' if predicted == true_label else '✗'
        if predicted == true_label: correct += 1
        print(f'{review[:48]:<50} {true_label:<18} {predicted:<18} {ok}')
    print(f'\nAccuracy: {correct}/{len(reviews_with_labels)} = {correct/len(reviews_with_labels):.0%}')
except Exception as e:
    print(f'Pipeline unavailable: {e}')
    print('Expected: ~80–90% accuracy. Pricing is easiest; product quality hardest to distinguish from others.')

---
## Exercise 12.4 — Sentence Similarity via [CLS] Embeddings

**Task:** Extract [CLS] embeddings for 20 sentences, find top-5 most similar pairs.

**Key concept:** BERT's [CLS] token is designed as a sentence-level representation. Cosine similarity between [CLS] vectors captures semantic relatedness. Sentences on the same topic should cluster together.

In [ ]:
sentences = [
    'Deep learning models require large amounts of training data.',
    'Neural networks learn hierarchical representations from examples.',
    'The stock market fell sharply after the inflation report.',
    'Interest rates rose following the central bank announcement.',
    'Scientists discovered a new exoplanet in the Milky Way galaxy.',
    'The telescope captured images of distant stars and nebulae.',
    'The chef prepared a delicious pasta with fresh ingredients.',
    'Italian cuisine is famous for its rich flavors and olive oil.',
    'Climate change is causing more frequent extreme weather events.',
    'Carbon emissions must be reduced to limit global warming.',
    'The soccer team won the championship after a dramatic final.',
    'Athletes trained for months to prepare for the Olympic games.',
    'Python is a popular language for machine learning projects.',
    'Transformers have revolutionized natural language processing.',
    'The museum houses a collection of Renaissance paintings.',
    'Modern art challenges traditional notions of beauty.',
    'Vaccines have dramatically reduced childhood mortality rates.',
    'Antibiotic resistance is a growing public health concern.',
    'The novel explores themes of identity and belonging.',
    'Poetry uses rhythm and metaphor to convey emotion.',
]

try:
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    model     = AutoModel.from_pretrained('bert-base-uncased')
    model.eval()

    def get_cls_embedding(text):
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # [CLS] token

    embeddings = np.array([get_cls_embedding(s) for s in sentences])
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, -1)  # exclude self-similarity

    # Find top-5 pairs
    pairs = [(sim_matrix[i, j], i, j)
             for i in range(len(sentences))
             for j in range(i+1, len(sentences))]
    top5 = sorted(pairs, reverse=True)[:5]

    print('Top 5 most similar sentence pairs:')
    for rank, (score, i, j) in enumerate(top5, 1):
        print(f'\n#{rank} (cosine={score:.4f})')
        print(f'  A: {sentences[i]}')
        print(f'  B: {sentences[j]}')

except Exception as e:
    print(f'BERT unavailable: {e}')
    print('Expected: Top pairs should be within-topic (e.g., two AI sentences, two climate sentences).')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 12.1 | BART = faithful/longer; PEGASUS = abstractive/shorter; choose by use case |
| 12.2 | BERT=WordPiece (##); GPT-2=BPE (Ġ); T5=SentencePiece (▁) — same idea, different notations |
| 12.3 | Zero-shot via NLI: no fine-tuning needed; works best when labels are natural phrases |
| 12.4 | [CLS] cosine similarity clusters semantically related sentences; within-topic pairs score highest |

---
*NLP Course 2027 — Week 06*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**